# 🔥 PyTorch — Deep Learning
## Python Ecosystem Tutorial Series — Module 3 of 18

**Author:** Himanshu Goel | [himanshugoel.github.io](https://himanshugoel.github.io)

---

| | |
|---|---|
| **Library** | 🔥 PyTorch |
| **Domain** | Deep Learning |
| **Dataset** | Iris (neural network) |
| **Module** | 3 of 18 |

**What you will learn:**

1. What PyTorch is and why it exists
2. Core concepts and data structures
3. Hands-on code with real data
4. Visualisations and interpretation
5. When to use it and alternatives

```bash
# Install required libraries
pip install pytorch
```

## Quick Reference Card

| Code | What it does |
|------|--------------|
| `torch.tensor()` | Create a tensor |
| `tensor.backward()` | Compute gradients |
| `nn.Module` | Base class for models |
| `optimizer.step()` | Update weights |
| `model.eval()` | Inference mode |

# 3. 🔥 PyTorch — Deep Learning
> **Python + PyTorch = Deep Learning**

PyTorch is Facebook's deep learning framework. It uses dynamic computation graphs —
great for research and flexible model building.

**Key concepts:** Tensors, autograd, nn.Module, DataLoader, training loop, GPU support

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# ── Understanding Tensors (the building block) ────────────────────────────────
print("\n── Tensor basics ──")
t1 = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
t2 = torch.randn(2, 2)
print(f"Tensor t1:\n{t1}")
print(f"t1 shape: {t1.shape}, dtype: {t1.dtype}")
print(f"t1 + t2:\n{(t1+t2).round(3)}")

# ── Autograd: automatic differentiation ──────────────────────────────────────
print("\n── Autograd (how backprop works) ──")
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x + 1   # y = (x+1)^2
y.backward()          # compute dy/dx
print(f"y = x² + 2x + 1  at x=3")
print(f"dy/dx = 2x + 2 = {x.grad.item()}")   # should be 8

In [ ]:
# ── Build a Neural Network for classification ─────────────────────────────────
# Using the Iris dataset (same as sklearn section)
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

iris = load_iris()
X, y = iris.data.astype(np.float32), iris.target

sc = StandardScaler()
X_scaled = sc.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Convert to PyTorch tensors
X_tr = torch.FloatTensor(X_train)
y_tr = torch.LongTensor(y_train)
X_te = torch.FloatTensor(X_test)
y_te = torch.LongTensor(y_test)

# ── Define the model ──────────────────────────────────────────────────────────
class IrisNet(nn.Module):
    """3-layer neural network: 4→64→32→3"""
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(4, 64),    # input layer: 4 features → 64 neurons
            nn.ReLU(),           # activation function
            nn.Dropout(0.2),     # regularisation: randomly drop 20% of neurons
            nn.Linear(64, 32),   # hidden layer
            nn.ReLU(),
            nn.Linear(32, 3),    # output layer: 3 classes
        )

    def forward(self, x):
        return self.network(x)

model = IrisNet().to(device)
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ── Training loop ─────────────────────────────────────────────────────────────
criterion = nn.CrossEntropyLoss()          # loss function
optimizer = optim.Adam(model.parameters(), lr=0.001)   # optimizer

train_losses, test_accs = [], []

for epoch in range(200):
    # ── Training step ──
    model.train()                          # put model in training mode
    optimizer.zero_grad()                  # clear previous gradients
    outputs = model(X_tr.to(device))       # forward pass
    loss    = criterion(outputs, y_tr.to(device))  # compute loss
    loss.backward()                        # backpropagation
    optimizer.step()                       # update weights
    train_losses.append(loss.item())

    # ── Evaluation step ──
    model.eval()                           # put model in evaluation mode
    with torch.no_grad():                  # no gradients needed for inference
        test_out = model(X_te.to(device))
        preds    = test_out.argmax(dim=1)
        acc      = (preds == y_te.to(device)).float().mean().item()
        test_accs.append(acc)

print(f"Final accuracy: {test_accs[-1]:.3f}")
print(f"Final loss:     {train_losses[-1]:.4f}")

# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses, color="#E74C3C", lw=2)
axes[0].set_title("Training Loss", fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].grid(True, alpha=0.3)

axes[1].plot(test_accs, color="#27AE60", lw=2)
axes[1].set_title("Test Accuracy", fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy")
axes[1].axhline(0.95, c="r", ls="--", alpha=0.5, label="95%"); axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("PyTorch — Neural Network Training on Iris", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("pytorch_iris.png", dpi=120, bbox_inches="tight")
plt.show()

## Deep Dive: PyTorch

### Dynamic Computation Graphs
PyTorch builds the computation graph **as code runs** — you can use if/else, loops, and debug with normal print statements. Old TensorFlow required defining the whole graph upfront, then running it separately.

### Autograd: How Backpropagation Works
PyTorch tracks every tensor operation:
```python
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x + 1   # PyTorch records: "y = x^2 + 2x + 1"
y.backward()           # automatically differentiates
print(x.grad)          # 8.0 — because dy/dx = 2x+2 = 2(3)+2 = 8
```
This is the engine behind all of deep learning.

### The 5-Step Training Loop
Every neural network trains with this exact pattern:
```python
optimizer.zero_grad()           # 1. clear old gradients
output = model(X)               # 2. forward pass (predict)
loss = criterion(output, y)     # 3. compute loss (error)
loss.backward()                  # 4. backprop (compute gradients)
optimizer.step()                 # 5. update weights
```

### Why Dropout Helps
During training, 20% of neurons are randomly zeroed each pass. This forces the network to learn redundant representations — no single neuron can be relied upon. The result: better generalisation to new data.

### model.train() vs model.eval()
```python
model.train()   # enables Dropout and BatchNorm training mode
model.eval()    # disables them for inference
with torch.no_grad():  # skips gradient computation (faster, less memory)
    predictions = model(X_test)
```


## ✅ Key Takeaways — 🔥 PyTorch

1. PyTorch's dynamic graphs make debugging as easy as normal Python
2. Autograd computes all derivatives automatically — no manual math needed
3. The 5-step training loop (zero_grad → forward → loss → backward → step) is universal
4. Always call model.eval() and torch.no_grad() during inference

---
*Next: Continue to Module 4 of 18 in the Python Ecosystem Tutorial Series*  
*Portfolio: [himanshugoel.github.io](https://himanshugoel.github.io)*